In [61]:
%matplotlib inline

import numpy as np
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l
from IPython.display import display

TRAIN_PATH = '../data/california-house-prices/train.csv'
TEST_PATH = '../data/california-house-prices/test.csv'

train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

In [62]:
print(train_data.shape)
print(test_data.shape)

(47439, 41)
(31626, 40)


In [63]:
TARGET = 'Sold Price'
all_features = pd.concat(
    [
        train_data.drop(columns=['Id', TARGET]),
        test_data.drop(columns=['Id'])
    ],
    ignore_index=True
)

In [64]:
print(all_features.shape)

(79065, 40)


# 特征工程

In [65]:
numeric_features = all_features.dtypes[all_features.dtypes != 'object'].index 

# 数据标准化
all_features[numeric_features] = all_features[numeric_features].apply(
    lambda x: (x - x.mean()) / x.std()
)
all_features[numeric_features] = all_features[numeric_features].fillna(0)

In [66]:
all_features.drop(columns=['Summary','Address'],inplace=True)
cat_features = all_features.select_dtypes(exclude='number').columns

In [67]:
all_features[cat_features].nunique()

Type                     174
Heating                 2658
Cooling                  909
Parking                 9911
Bedrooms                 277
Region                  1258
Elementary School       3567
Middle School            808
High School              921
Flooring                1738
Heating features        1761
Cooling features         594
Appliances included    11288
Laundry features        3029
Parking features        9693
Listed On               2815
Last Sold On            6948
City                    1122
State                      2
dtype: int64

In [68]:
all_features = pd.get_dummies(
    all_features, dummy_na=True, dtype=np.float32
)

In [74]:
n_train = train_data.shape[0]
train_features = torch.tensor(
    all_features.iloc[:n_train].to_numpy(dtype=np.float32),
    dtype=torch.float32
)
test_features = torch.tensor(
    all_features.iloc[n_train:].to_numpy(dtype=np.float32),
    dtype=torch.float32
)
train_labels = torch.tensor(
    train_data[TARGET].to_numpy(dtype=np.float32).reshape(-1, 1),
    dtype=torch.float32
)

# 简单 MLP

大数据集先使用一次训练集/验证集划分，不做 K 折验证。标签使用 `log1p`，让房价回归训练更加稳定。

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(42)

labels_log = torch.log1p(train_labels)
full_dataset = TensorDataset(train_features, labels_log)
valid_size = int(len(full_dataset) * 0.1)
train_size = len(full_dataset) - valid_size
train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 256
train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_iter = DataLoader(valid_dataset, batch_size=batch_size)

def get_mlp(in_features):
    return nn.Sequential(
        nn.Linear(in_features, 64),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

net = get_mlp(train_features.shape[1])
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
net

In [ ]:
def train_one_epoch(net, data_iter, optimizer, loss_fn, max_batches=None):
    net.train()
    loss_sum, sample_count = 0.0, 0

    for batch_index, (X, y) in enumerate(data_iter):
        if max_batches is not None and batch_index >= max_batches:
            break

        optimizer.zero_grad()
        predictions = net(X)
        loss = loss_fn(predictions, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * X.shape[0]
        sample_count += X.shape[0]

    return loss_sum / sample_count

@torch.no_grad()
def evaluate(net, data_iter, loss_fn, max_batches=None):
    net.eval()
    loss_sum, sample_count = 0.0, 0

    for batch_index, (X, y) in enumerate(data_iter):
        if max_batches is not None and batch_index >= max_batches:
            break

        loss = loss_fn(net(X), y)
        loss_sum += loss.item() * X.shape[0]
        sample_count += X.shape[0]

    return loss_sum / sample_count

In [ ]:
# 只跑少量 batch，确认前向传播、反向传播和验证流程能够工作
smoke_train_loss = train_one_epoch(
    net, train_iter, optimizer, loss_fn, max_batches=5
)
smoke_valid_loss = evaluate(
    net, valid_iter, loss_fn, max_batches=2
)

print(f'smoke train log-MSE: {smoke_train_loss:.6f}')
print(f'smoke valid log-MSE: {smoke_valid_loss:.6f}')

# smoke test 通过后再正式训练，例如：
# for epoch in range(20):
#     train_loss = train_one_epoch(net, train_iter, optimizer, loss_fn)
#     valid_loss = evaluate(net, valid_iter, loss_fn)
#     print(f'epoch {epoch + 1}: train={train_loss:.6f}, valid={valid_loss:.6f}')